# Лабораторна робота №02
## LLM-екстракція знань, векторний пошук та GraphRAG

**Предметна область:** Українська революція 1917–1921 рр. (Українська Центральна Рада, Гетьманат, Директорія УНР, ЗУНР та ключові постаті)
**Корпус:** статті української Вікіпедії
**LLM / ембединги:** Gemini API (`google-genai`), Ollama bge-m3 для ембедингу, `numpy` для векторного сховища, `networkx` + `python-louvain` для графа

Усі відповіді LLM і ембединги кешуються в папці `cache/`, тому після збою або перезапуску повторні виклики API не витрачають ліміти


## Налаштування

In [2]:
import os, re, json, time, hashlib, random
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Literal, Optional

import numpy as np
import requests
from dotenv import load_dotenv
from pydantic import BaseModel
from google import genai
from google.genai import types

load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
assert API_KEY, "GEMINI_API_KEY not found in .env"
client = genai.Client(api_key=API_KEY)

LLM_MODEL   = os.getenv("LLM_MODEL", "gemini-3.5-flash-lite")
OLLAMA_URL = "http://localhost:11434/api/embed"
OLLAMA_EMBED_MODEL = "bge-m3"
EMBED_MODEL = f"ollama-{OLLAMA_EMBED_MODEL}"
EMBED_DIM   = 1024
MIN_INTERVAL = float(os.getenv("MIN_INTERVAL", "4.5"))

CACHE = Path("cache"); CACHE.mkdir(exist_ok=True)
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
print("LLM:", LLM_MODEL, "| Embeddings:", EMBED_MODEL)

LLM: gemini-3.5-flash-lite | Embeddings: ollama-bge-m3


In [ ]:
_last_call = 0.0
def _throttle():
    global _last_call
    wait = MIN_INTERVAL - (time.time() - _last_call)
    if wait > 0:
        time.sleep(wait)
    _last_call = time.time()

def _with_retry(fn, max_tries=6):
    for attempt in range(max_tries):
        try:
            _throttle()
            return fn()
        except Exception as e:
            msg = str(e)
            transient = any(s in msg for s in ("429", "500", "503", "504", "RESOURCE_EXHAUSTED", "UNAVAILABLE", "DEADLINE"))
            if not transient or attempt == max_tries - 1:
                raise
            delay = min(60, 5 * 2 ** attempt)
            print(f"\tТимчасова помилка API ({msg}), повтор через {delay} с")
            time.sleep(delay)

def _key(*parts):
    return hashlib.sha256("||".join(map(str, parts)).encode("utf-8")).hexdigest()[:24]

def llm_json(prompt, schema, system=None):
    f = CACHE / f"llm_{_key(LLM_MODEL, schema.__name__, system, prompt)}.json"
    if f.exists():
        return schema.model_validate_json(f.read_text(encoding="utf-8"))
    def call():
        return client.models.generate_content(
            model=LLM_MODEL, contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=system,
                response_mime_type="application/json",
                response_schema=schema,
                temperature=0.0))
    resp = _with_retry(call)
    obj = resp.parsed if getattr(resp, "parsed", None) is not None else schema.model_validate_json(resp.text)
    f.write_text(obj.model_dump_json(), encoding="utf-8")
    return obj

def llm_text(prompt, system=None, temperature=0.2):
    f = CACHE / f"txt_{_key(LLM_MODEL, system, temperature, prompt)}.txt"
    if f.exists():
        return f.read_text(encoding="utf-8")
    def call():
        return client.models.generate_content(
            model=LLM_MODEL, contents=prompt,
            config=types.GenerateContentConfig(system_instruction=system, temperature=temperature))
    text = _with_retry(call).text
    f.write_text(text, encoding="utf-8")
    return text

def embed_texts(texts, task_type, batch=16):
    f = CACHE / f"emb_{_key(EMBED_MODEL, task_type, *texts)}.npy"
    if f.exists():
        return np.load(f)
    vecs = []
    for i in range(0, len(texts), batch):
        r = requests.post(OLLAMA_URL, json={"model": OLLAMA_EMBED_MODEL, "input": texts[i:i + batch]}, timeout=300)
        r.raise_for_status()
        vecs.extend(r.json()["embeddings"])
    arr = np.array(vecs, dtype=np.float32)
    arr /= np.linalg.norm(arr, axis=1, keepdims=True)
    np.save(f, arr)
    return arr

## 1. Предметна область, корпус і запитання

**Область:** Українська революція 1917–1921 рр. Тема добре підходить для графа знань оскільки має багато осіб, організацій, подій і місць з чіткими зв'язками

**Джерело даних:** Вікіпедія, 12 статей. З кожної беремо початок статті (до `DOC_CHAR_LIMIT` символів) без розділів «Примітки», «Література», «Посилання» тощо. Це зменшує кількість викликів API і лишає зв'язний текст. Посилання на кожну статтю зберігаються в `corpus/sources.json`.

In [ ]:
ARTICLES = [
    "Українська революція",
    "Українська Центральна Рада",
    "Українська Держава (1918)",
    "Директорія УНР",
    "Західноукраїнська Народна Республіка",
    "Акт злуки",
    "Симон Петлюра",
    "Михайло Грушевський",
    "Павло Скоропадський",
    "Володимир Винниченко",
    "Бій під Крутами",
    "Перший Зимовий похід",
]
DOC_CHAR_LIMIT = 20000

WIKI_API = "https://uk.wikipedia.org/w/api.php"
HEADERS = {"User-Agent": "sw lab-2"}
CUT_SECTIONS = {"Примітки", "Література", "Посилання", "Див. також", "Джерела", "Зовнішні посилання",
                "Виноски", "Галерея", "Джерела та література", "Бібліографія", "Примітки та джерела"}

def _wiki_query(**params):
    r = requests.get(WIKI_API, params={"format": "json", **params}, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

def fetch_wiki_article(title):
    data = _wiki_query(action="query", prop="extracts|info", explaintext=1, redirects=1, titles=title, inprop="url")
    page = next(iter(data["query"]["pages"].values()))
    if "missing" in page:
        return None
    return {"title": page["title"], "url": page.get("fullurl", ""), "text": page.get("extract", "")}

def search_wiki_title(query):
    data = _wiki_query(action="query", list="search", srsearch=query, srlimit=1)
    hits = data["query"]["search"]
    return hits[0]["title"] if hits else None

def clean_wiki_text(text, limit):
    paras, total = [], 0
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        m = re.match(r"^=+\s*(.*?)\s*=+$", line)
        # заголовок розділу
        if m:                                  
            if m.group(1) in CUT_SECTIONS:
                break
            line = m.group(1) + "."
        if total + len(line) > limit and paras:
            break
        paras.append(line); total += len(line) + 1
    return "\n".join(paras)

corpus_dir = Path("corpus"); corpus_dir.mkdir(exist_ok=True)
sources_file = corpus_dir / "sources.json"

if sources_file.exists():
    DOCS = json.loads(sources_file.read_text(encoding="utf-8"))
    for d in DOCS:
        d["text"] = (corpus_dir / f"{d['id']}.txt").read_text(encoding="utf-8")
    print("Корпус завантажено з диска:", len(DOCS), "документів")
else:
    DOCS = []
    for title in ARTICLES:
        art = fetch_wiki_article(title)
        if art is None:
            alt = search_wiki_title(title)
            art = fetch_wiki_article(alt) if alt else None
        if art is None or not art["text"].strip():
            print(f"  пропущено: {title}")
            continue
        text = clean_wiki_text(art["text"], DOC_CHAR_LIMIT)
        doc = {"id": f"doc{len(DOCS)+1:02d}", "title": art["title"], "url": art["url"], "text": text}
        DOCS.append(doc)
        (corpus_dir / f"{doc['id']}.txt").write_text(text, encoding="utf-8")
        time.sleep(0.5)
    sources_file.write_text(json.dumps([{k: v for k, v in d.items() if k != "text"} for d in DOCS],
                                       ensure_ascii=False, indent=2), encoding="utf-8")

total_words = 0
print(f"{'id':6} {'слів':>6}  назва")
for d in DOCS:
    w = len(d["text"].split()); total_words += w
    print(f"{d['id']:6} {w:>6}  {d['title']}")
print(f"\nДокументів: {len(DOCS)}, слів разом: {total_words}")

Корпус завантажено з диска: 12 документів
id       слів  назва
doc01    2400  Українська революція (1917—1921)
doc02    1348  Українська Центральна Рада
doc03    2159  Українська Держава
doc04    2077  Директорія Української Народної Республіки
doc05    2679  Західноукраїнська Народна Республіка
doc06    1381  Акт Злуки
doc07    2638  Петлюра Симон Васильович
doc08    2627  Грушевський Михайло Сергійович
doc09    2446  Скоропадський Павло Петрович
doc10    2611  Винниченко Володимир Кирилович
doc11    2686  Бій під Крутами (1918)
doc12    1046  Перший зимовий похід

Документів: 12, слів разом: 26098


### Запитання до тексту

In [5]:
QUESTIONS = [
    {"id": "Q1", "kind": "simple", "text": "Коли і де було проголошено Акт злуки УНР і ЗУНР?"},
    {"id": "Q2", "kind": "simple", "text": "Хто очолював Українську Центральну Раду?"},
    {"id": "Q3", "kind": "simple", "text": "Коли і де відбувся бій під Крутами?"},
    {"id": "Q4", "kind": "global", "text": "Хто є ключовими фігурами Української революції 1917–1921 рр. і як вони пов'язані між собою?"},
    {"id": "Q5", "kind": "global", "text": "Які причини поразки українських державних утворень 1917–1921 рр. згадуються в текстах найчастіше?"},
    {"id": "Q6", "kind": "global", "text": "Які політичні сили та держави протистояли УНР і з ким вона укладала союзи?"},
]
for q in QUESTIONS:
    print(q["id"], f"[{q['kind']}]", q["text"])

Q1 [simple] Коли і де було проголошено Акт злуки УНР і ЗУНР?
Q2 [simple] Хто очолював Українську Центральну Раду?
Q3 [simple] Коли і де відбувся бій під Крутами?
Q4 [global] Хто є ключовими фігурами Української революції 1917–1921 рр. і як вони пов'язані між собою?
Q5 [global] Які причини поразки українських державних утворень 1917–1921 рр. згадуються в текстах найчастіше?
Q6 [global] Які політичні сили та держави протистояли УНР і з ким вона укладала союзи?


## 2. LLM-екстракція структурованих фактів

### Схема сутностей
- **4 типи сутностей:** `Person`, `Organization` (організація, держава, партія, військова частина), `Event` (подія, битва, договір, акт), `Location`
- **8 типів зв'язків:** `LEADER_OF`, `MEMBER_OF`, `ALLIED_WITH`, `OPPOSED`, `PARTICIPATED_IN`, `FOUNDED`, `SUCCEEDED`, `LOCATED_IN`
- **Атрибути:** у сутності: короткий опис; у зв'язку: дата та `evidence`: дослівна цитата з тексту. Цитата потрібна, щоб легко перевіряти факт вручну і автоматично ловити вигадки: якщо цитати немає в уривку, факт підозрілий

Отже **Факт** це дані виду: суб'єкт → тип зв'язку → об'єкт (+ дата + цитата)

In [ ]:
EntityType   = Literal["Person", "Organization", "Event", "Location"]
RelationType = Literal["LEADER_OF", "MEMBER_OF", "ALLIED_WITH", "OPPOSED",
                       "PARTICIPATED_IN", "FOUNDED", "SUCCEEDED", "LOCATED_IN"]

class Entity(BaseModel):
    name: str
    type: EntityType
    description: str

class Relation(BaseModel):
    source: str
    relation: RelationType
    target: str
    date: Optional[str] = None
    evidence: str

class Extraction(BaseModel):
    entities: List[Entity]
    relations: List[Relation]

### Розбиття на чанки
Одна функція використовується і для екстракції, і для векторного пошуку. 

Чанки розбиваються по реченнях, щоб не обрізати думку по середині. Також для векторного пошуку додається перекриття

In [6]:
def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?…])\s+|\n+", text) if s.strip()]

def chunk_text(text, size=800, overlap=0):
    chunks, cur, cur_len = [], [], 0
    for s in split_sentences(text):
        if cur and cur_len + len(s) + 1 > size:
            chunks.append(" ".join(cur))
            keep, kl = [], 0
            for t in reversed(cur):
                if kl + len(t) > overlap:
                    break
                keep.insert(0, t); kl += len(t) + 1
            cur, cur_len = keep, kl
        cur.append(s); cur_len += len(s) + 1
    if cur:
        chunks.append(" ".join(cur))
    return chunks

### Запуск екстракції
Використовуються великі чанки по ≈ 3500 символів. Відповіді хешуються

In [83]:
EXTRACT_CHUNK = 3500
extract_units = [(d, i, c) for d in DOCS for i, c in enumerate(chunk_text(d["text"], EXTRACT_CHUNK, 0))]
print("Чанків для екстракції:", len(extract_units))

SYSTEM_EXTRACT = """Ти — система вилучення знань з історичних текстів про Українську революцію 1917–1921 рр.
Правила:
1. Використовуй ЛИШЕ інформацію з наданого уривка. Нічого не додавай зі своїх знань і не домислюй.
2. Назви сутностей — повна канонічна форма в називному відмінку («Симон Петлюра», а не «Петлюрою»; «Українська Центральна Рада», а не «Рада»), якщо повна назва є в тексті. Займенники («він», «вона») розв'язуй за контекстом.
3. У relations поля source і target мають ТОЧНО збігатися з name якоїсь сутності з entities.
4. Типи зв'язків (напрямок source -> target):
   LEADER_OF: особа очолювала організацію/подію; MEMBER_OF: особа чи організація входила до складу організації;
   ALLIED_WITH: союзники/договір про співпрацю; OPPOSED: воювали або протистояли;
   PARTICIPATED_IN: особа/організація брала участь у події; FOUNDED: створив/проголосив організацію чи державу;
   SUCCEEDED: source став наступником/змінив target; LOCATED_IN: подія/організація відбулася чи розташована в місці.
5. evidence — ДОСЛІВНА цитата з уривка (до 30 слів), що підтверджує зв'язок. date — як у тексті, або null.
6. Якщо в уривку немає фактів за цією схемою — поверни порожні списки."""

def build_extract_prompt(doc_title, chunk):
    return f"Стаття: «{doc_title}»\n\nУривок:\n\"\"\"\n{chunk}\n\"\"\"\n\nВиділи сутності та зв'язки за схемою."

raw_extractions, failed = [], []
for n, (d, i, chunk) in enumerate(extract_units, 1):
    try:
        ext = llm_json(build_extract_prompt(d["title"], chunk), Extraction, SYSTEM_EXTRACT)
        raw_extractions.append({"doc": d, "chunk_idx": i, "chunk": chunk, "ext": ext})
    except Exception as e:
        failed.append((d["id"], i, str(e)[:150]))
        print(f" Помилка {d['id']} чанк {i}: {str(e)[:120]}")
    if n % 5 == 0 or n == len(extract_units):
        print(f"  оброблено {n}/{len(extract_units)}")
print("Успішно:", len(raw_extractions), "| помилок:", len(failed))

Чанків для екстракції: 62
  оброблено 5/62
  оброблено 10/62
  оброблено 15/62
  оброблено 20/62
  оброблено 25/62
  оброблено 30/62
  оброблено 35/62
  оброблено 40/62
  оброблено 45/62
  оброблено 50/62
  оброблено 55/62
  оброблено 60/62
  оброблено 62/62
Успішно: 62 | помилок: 0


### Збірка фактів
1. **Нормалізація імен:** прибираємо зайві пробіли та застосовуємо словник `ALIASES` щоб прибрати дублювання
2. **Автоматична перевірка цитат:** наявність цитати перевіряється з точністю до регістру/пробілів/лапок
3. **Дедуплікація:** однакові трійки (суб'єкт, зв'язок, об'єкт) зводяться в один факт і підраховуються кількість згадок

In [16]:
ALIASES = {
    "УЦР": "Українська Центральна Рада",
    "Центральна Рада": "Українська Центральна Рада",
    "УНР": "Українська Народна Республіка",
    "ЗУНР": "Західноукраїнська Народна Республіка",
    "УГА": "Українська Галицька армія",
    "Петлюра": "Симон Петлюра",
    "Грушевський": "Михайло Грушевський",
    "Скоропадський": "Павло Скоропадський",
    "Винниченко": "Володимир Винниченко",
    "Директорія": "Директорія УНР",
    "Муравйов": "Михаїл Муравйов",
    "Муравйов": "Михайло Муравйов",
    "Омельченко": "Андрій Омельченко",
    "Галицька армія": "Українська Галицька армія",
    "Гайдамацький кіш": "Гайдамацький кіш Слобідської України",
    "Добровольча армія": "Добровольча армія Денікіна",
    "Військовий Комітет": "Український генеральний військовий комітет",
    "Студентський Курінь": "Студентський курінь Січових Стрільців",
    "Студентський курінь": "Студентський курінь Січових Стрільців",
    "Народний секретаріат": "Народний Секретаріат УНР",
    "Державний Секретаріат": "Державний секретаріат ЗУНР",
    "Рада народних комісарів": "Рада Народних Комісарів РСФРР",
    "Георгій": "Георгій Львов",
    "Національна Рада": "Українська Національна Рада",
    "Національна Рада": "Українська Національна Рада ЗУНР",
    "Українська Національна Рада": "Українська Національна Рада ЗУНР",
    "I Всеукраїнський військовий з'їзд": "Перший Всеукраїнський військовий з'їзд",
    "Більшовики": "російські більшовики",
    "Рада Міністрів": "Рада народних міністрів"
}

In [17]:
def clean_name(name):
    name = re.sub(r"\s+", " ", name).strip(" \"'«».,;")
    return ALIASES.get(name, name)

def norm_text(s):
    return re.sub(r"[\s\"'«»“”„‘’`´]+", " ", s.lower()).strip()

facts, seen = [], {}
entity_info = defaultdict(lambda: {"types": Counter(), "descriptions": [], "mentions": 0})

for r in raw_extractions:
    d, i, chunk, ext = r["doc"], r["chunk_idx"], r["chunk"], r["ext"]
    local_types = {}
    for e in ext.entities:
        nm = clean_name(e.name)
        local_types[nm] = e.type
        info = entity_info[nm]
        info["types"][e.type] += 1; info["mentions"] += 1
        if e.description and e.description not in info["descriptions"]:
            info["descriptions"].append(e.description)
    for rel in ext.relations:
        s, t = clean_name(rel.source), clean_name(rel.target)
        if not s or not t or s == t:
            continue
        key = (s.lower(), rel.relation, t.lower())
        ev_ok = norm_text(rel.evidence) in norm_text(chunk) if rel.evidence else False
        if key in seen:
            seen[key]["mentions"] += 1
            continue
        f = {"id": len(facts) + 1, "subject": s, "subject_type": local_types.get(s, "Unknown"),
             "predicate": rel.relation, "object": t, "object_type": local_types.get(t, "Unknown"),
             "date": rel.date, "evidence": rel.evidence, "evidence_ok": ev_ok,
             "doc_id": d["id"], "doc_title": d["title"], "chunk_idx": i, "mentions": 1}
        facts.append(f); seen[key] = f

ENTITIES = {nm: {"type": info["types"].most_common(1)[0][0], "descriptions": info["descriptions"], "mentions": info["mentions"]}
            for nm, info in entity_info.items()}
# для сутностей без власного опису (з'явилися лише у зв'язках)
for f in facts:
    for nm, tp in ((f["subject"], f["subject_type"]), (f["object"], f["object_type"])):
        ENTITIES.setdefault(nm, {"type": tp, "descriptions": [], "mentions": 1})

(OUT / "facts.json").write_text(json.dumps(facts, ensure_ascii=False, indent=2), encoding="utf-8")
(OUT / "entities.json").write_text(json.dumps(ENTITIES, ensure_ascii=False, indent=2), encoding="utf-8")

print("Фактів:", len(facts), "| сутностей:", len(ENTITIES))
print("Типи зв'язків:", dict(Counter(f["predicate"] for f in facts)))
print("Типи сутностей:", dict(Counter(e["type"] for e in ENTITIES.values())))
bad_ev = sum(not f["evidence_ok"] for f in facts)
print(f"Цитата НЕ знайдена в тексті (підозра на вигадку): {bad_ev} з {len(facts)} ({100*bad_ev/max(1,len(facts)):.0f}%)")

Фактів: 300 | сутностей: 420
Типи зв'язків: {'LEADER_OF': 68, 'FOUNDED': 20, 'ALLIED_WITH': 19, 'MEMBER_OF': 98, 'OPPOSED': 30, 'PARTICIPATED_IN': 27, 'LOCATED_IN': 32, 'SUCCEEDED': 6}
Типи сутностей: {'Organization': 153, 'Person': 197, 'Event': 14, 'Location': 49, 'Unknown': 7}
Цитата НЕ знайдена в тексті (підозра на вигадку): 27 з 300 (9%)


### Перевірка на можливі повторення

In [82]:
names = sorted(ENTITIES, key=len)
cands = []
for a_i, a in enumerate(names):
    for b in names[a_i + 1:]:
        if ENTITIES[a]["type"] == ENTITIES[b]["type"] and set(a.lower().split()) < set(b.lower().split()):
            cands.append((a, b))
print("Можливі дублі:", len(cands))
for a, b in cands:
    print(f"  \"{a}\": \"{b}\"")

Можливі дублі: 13
  "УСРР": "Раднарком УСРР"
  "РСФРР": "Рада Народних Комісарів РСФРР"
  "Василь": "Василь Панейко"
  "Василь": "Василь Стефаник"
  "Громада": "Українська Народна Громада"
  "Громада": "Чорноморська вільна громада"
  "Громада": "Українська студентська громада"
  "Раднарком": "Раднарком УСРР"
  "Раднарком": "Раднарком Росії"
  "Українська академія наук": "Українська вільна академія наук США"
  "Українська Народна Республіка": "Українська Народна Республіка Рад"
  "Всеукраїнський військовий з'їзд": "II Всеукраїнський військовий з'їзд"
  "Всеукраїнський військовий з'їзд": "Перший Всеукраїнський військовий з'їзд"


### Ручна перевірка 20 випадкових фактів
Відбір 20 випадкових фактів та створення `outputs/verification.csv` файлу, для вказування коректності факту та, якщо є помилка, - типу помилки

In [19]:
SAMPLE_SIZE = 20
sample = random.Random(42).sample(facts, min(SAMPLE_SIZE, len(facts)))
import csv
with open(OUT / "verification.csv", "w", newline="", encoding="utf-8-sig") as fh:
    w = csv.writer(fh)
    w.writerow(["fact_id", "doc", "subject", "predicate", "object", "date", "evidence", "evidence_ok", "verdict", "error_type", "comment"])
    for f in sample:
        w.writerow([f["id"], f["doc_title"], f["subject"], f["predicate"], f["object"], f["date"] or "",
                    f["evidence"], f["evidence_ok"], "", "", ""])

Перед запуском наступного блоку було вручну перевірено кожен факт

In [21]:
import pandas as pd
v = pd.read_csv(OUT / "verification.csv", encoding="utf-8-sig", sep=';').fillna("")
done = v[v["verdict"].isin(["ok", "error"])]
if len(done) < len(v):
    print(f"Заповнено {len(done)} з {len(v)} рядків. Заповніть колонку verdict у verification.csv.")
else:
    ok = (done["verdict"] == "ok").sum()
    print(f"Коректних фактів: {ok}/{len(done)} = {100*ok/len(done):.0f}%")
    print(done[done["verdict"] == "error"]["error_type"].value_counts().to_string())

Коректних фактів: 17/20 = 85%
error_type
неправильний зв'язок               1
incorrect object                   1
неправильний зв'язок або object    1


C:\Users\sedv2\AppData\Local\Temp\ipykernel_17652\365139405.py:2: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  v = pd.read_csv(OUT / "verification.csv", encoding="utf-8-sig", sep=';').fillna("")


### Приклади фактів

In [72]:
for f in random.Random(1).sample(facts, min(8, len(facts))):
    print(f"[{f['doc_title']}] {f['subject']} ({f['subject_type']}) —{f['predicate']}→ {f['object']} ({f['object_type']})  дата: {f['date']}")
    print(f"    «{f['evidence']}»  цитата в тексті: {f['evidence_ok']}")

[Директорія Української Народної Республіки] Панас Андрієвський (Person) —MEMBER_OF→ Директорія УНР (Organization)  дата: 4 травня 1919 року
    «Панас Андрієвський — 4 травня 1919 року вийшов зі складу Директорії УНР»  цитата в тексті: True
[Перший зимовий похід] Київська дивізія (Organization) —MEMBER_OF→ Перший зимовий похід (Event)  дата: None
    «до складу північної колони отамана Ю. Тютюнника — Київська та Волинська дивізії»  цитата в тексті: True
[Українська Центральна Рада] Українська Центральна Рада (Organization) —FOUNDED→ Генеральний Секретаріат (Organization)  дата: 15 (28) червня
    «По проголошенню автономії 15 (28) червня створений Генеральний Секретаріат.»  цитата в тексті: True
[Акт Злуки] В. Винниченко (Person) —MEMBER_OF→ Директорія УНР (Organization)  дата: None
    «членами Директорії УНР (О. Андрієвський, В. Винниченко, С. Петлюра, Ф. Швець)»  цитата в тексті: True
[Українська Держава] Україна (Organization) —OPPOSED→ російські більшовики (Organization)  дата: N

### Пропуски (recall)
Виконується відбір випадкового чанку, а потім вручну порівнюються отримані зв'язки та текст, який був представлений в чанку 

In [71]:
r = random.Random(7).choice(raw_extractions)
print("Стаття:", r["doc"]["title"], "| чанк", r["chunk_idx"], "\n")
print(r["chunk"][:3500])
print("\n--- Видобуті з цього чанка зв'язки ---")
for rel in r["ext"].relations:
    print(f"  {rel.source} {rel.relation} {rel.target} (Дата: {rel.date})")

Стаття: Західноукраїнська Народна Республіка | чанк 0 

Західноукраїнська Народна Республіка (за тодішнім правописом: Західно-Українська Народня Республіка, Західна-Українська Народня Република, ЗУНР, до 13 листопада 1918 — Украї́нська Держа́ва) — назва держави, що фактично проіснувала протягом 1918—1919 років на територіях заходу України зі столицею у Львові. Постала після Першої світової війни під час розпаду Австро-Угорщини. 19 жовтня 1918 року оголошена «Проклямація Української Національної Ради» про створення Української держави в українських етнічних землях Австро-Угорщини. 13 листопада 1918 року УНРада затвердила Конституційні основи новоствореної держави — «Тимчасовий основний закон про державну самостійність українських земель бувшої австро-угорської монархії», нова держава отримала назву «Західноукраїнська Народна Республіка». ЗУНР охоплювала територію заселену українцями: Східну Галичину, Буковину й Закарпаття. 1 листопада із заколоту всередині ЗУНР (у Львові), Польща першою

### Результат перевірки пропусків

Отримані факти мають помилки. Наприклад є одне повторення, яке зустрічається двічі:

* Степан Баран MEMBER_OF Український Народний Комітет (Дата: None)

Також неправильний об'єкт в членів комісії: Об'єктом обрано `Український Народний Комітет`, хоча вони були учасниками саме комісії, а не комітету (з даного фрагменту можна сказати лише це)

Деякі важливі факти були пропущені. Наприклад з отриманого чанку можна було б також додати такі факти:

| Суб'єкт | Зв'язок | Об'єкт |
|---------|---------|--------|
| УНРада | FOUNDED | ЗУНР |
| Країни Антанти | ALLIED_WITH | Польща |
| Польща | PARTICIPATED_IN | Польсько-українська війна |
| ЗУНР | PARTICIPATED_IN | Польсько-українська війна |
| Східна Галичина | LOCATED_IN | Польща |

## 3. Векторний пошук

**Чанкінг:** розмір приблизно 800 символів з перекриттям в 150 символів з розбиттям по реченнях.
Обґрунтування: чанк має вміщати одну завершену думку (1–2 абзаци); надто великі чанки "розмивають" вектор кількома темами, а надто малі втрачають контекст. Перекриття не дає розірвати зв'язок між сусідніми реченнями

**Сховище:** numpy-матриця; ембединги нормалізовані, тому косинусна подібність через скалярний добуток

In [22]:
VEC_CHUNK, VEC_OVERLAP = 800, 150
CHUNKS = []
for d in DOCS:
    for i, c in enumerate(chunk_text(d["text"], VEC_CHUNK, VEC_OVERLAP)):
        CHUNKS.append({"id": f"{d['id']}_{i:03d}", "doc_id": d["id"], "doc_title": d["title"], "text": c})
lens = [len(c["text"]) for c in CHUNKS]
print(f"Чанків: {len(CHUNKS)} | довжина (символів): мін {min(lens)}, середня {int(np.mean(lens))}, макс {max(lens)}")

CHUNK_EMB = embed_texts([c["text"] for c in CHUNKS], "RETRIEVAL_DOCUMENT")
print("Матриця ембедингів:", CHUNK_EMB.shape)

Чанків: 311 | довжина (символів): мін 176, середня 707, макс 799
Матриця ембедингів: (311, 1024)


### Порівняння keyword- і векторного пошуку

Додатково виконується пошук за підрядком

In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

def semantic_search(query, k=5):
    qv = embed_texts([query], "RETRIEVAL_QUERY")[0]
    sims = CHUNK_EMB @ qv
    idx = np.argsort(-sims)[:k]
    return [(CHUNKS[i], float(sims[i])) for i in idx]

_tfidf = TfidfVectorizer(lowercase=True, token_pattern=r"(?u)\b\w+\b")
_tfidf_matrix = _tfidf.fit_transform([c["text"] for c in CHUNKS])

def keyword_search(query, k=5):
    sims = linear_kernel(_tfidf.transform([query]), _tfidf_matrix).ravel()
    idx = np.argsort(-sims)[:k]
    return [(CHUNKS[i], float(sims[i])) for i in idx if sims[i] > 0]

def substring_search(query):
    q = query.lower()
    return [(c, 1.0) for c in CHUNKS if q in c["text"].lower()]

def show(label, results, n=3, width=800):
    print(f"  ── {label} ──")
    if not results:
        print("     (нічого не знайдено)")
    for rank, (c, s) in enumerate(results[:n], 1):
        print(f"     {rank}. [{s:.3f}] {c['doc_title']} ({c['id']}): {c['text'][:width].replace(chr(10), ' ')}…")

def compare(query, k=3, width=800):
    print(f"\nЗапит: «{query}»")
    show("TF-IDF", keyword_search(query, k), k, width)
    sub = substring_search(query)
    show(f"Підрядок (усього чанків: {len(sub)}, показано перші, у порядку документів)", sub, k, width)
    show("Семантичний", semantic_search(query, k), k, width)

In [ ]:
SEARCH_QUERIES = [
    "Як закінчилося існування першого українського парламенту після приходу німецьких військ?", #(перефразований запит)
    "Хто з політиків висував ідею соціалістичного уряду і критикував гетьмана?", #(перефразований запит)
    "29 січня 1918", # дата
    "Крути", # місце
]
for q in SEARCH_QUERIES:
    compare(q, k=5)


Запит: «Як закінчилося існування першого українського парламенту після приходу німецьких військ?»
  ── TF-IDF ──
     1. [0.111] Петлюра Симон Васильович (doc07_002): На цій посаді став одним із фундаторів українських збройних сил. Після приходу до влади гетьманців (30 квітня 1918 року) потрапив в опалу, а згодом став одним з організаторів повстання проти Скоропадського (листопад 1918 року). Після повалення останнього очолив війська та флот УНР та увійшов до складу Директорії. У цей період виступав за безкомпромісну боротьбу з більшовиками, через що постійно конфліктував із Володимиром Винниченком, який волів шукати компромісу з ворогом. Після поразки у війні з більшовиками підписав Варшавський договір (1920) із Польщею з метою залучити останню як союзника у війну з Радянською Росією та завдяки цьому відновити самостійність України.…
     2. [0.085] Скоропадський Павло Петрович (doc09_013): У січні 1918 року більшовики зайняли Київ, а Скоропадський переховувався від репресій. З поверн

### Висновки порівняння:

#### Запит: «Як закінчилося існування першого українського парламенту після приходу німецьких військ?»
Найкраще себе показав семантичний пошук, який дав найбільш чітку відповідь. Пошук за ключовим словом дав гірші результати, бо частина відповідей не сильно пов'язана з питанням, а також вищий `rank` не означає більш коректну відповідь. Пошук за підрядком нічого не знайшов

#### Запит: «Хто з політиків висував ідею соціалістичного уряду і критикував гетьмана?»
Пошук за ключовим словом по суті не дав відповіді, а лише описав гетьмана, а частина про соціалістичні ідеї ніяк не розглянута. Семантичний пошук дав відповідь про Петлюру, проте це згадано аж в 4 відповіді, інші ж відповіді пов'язані з діями гетьмана і тим, що йому протидіяли `соціалісти-федералісти, соціал-демократи` проте без чітких постатей. Пошук за підрядком нічого не знайшов

#### Запит: «29 січня 1918»
Пошук за ключовим пошуком має значний недолік, оскільки виконує окремо пошук за `29`, `січня` та `1918`, що повертає відповіді, що не пов'язані з реальною датою, за якою здійснювався пошук. Пошук за підрядком також не дав результатів, бо в документі не вказано чітко саме цю дату, а вказано `16 (29) або 17 (30) січня 1918 року` тобто включає ще й дати за іншим календарем. Семантичний пошук загалом в 4 з 5 відповідях посилається на коректні дані

#### Запит: «Крути»
За даним запитом всі способи пошуку показали себе однаково добре

За отриманими результатами у семантичного пошуку є одна особливість: оцінки відповідей дуже близькі, тому порядок майже випадковий

На розглянутих запитах, навіть з врахуванням пошуку за точною назвою та датою, семантичний пошук загалом показав себе найкраще

Пошук за ключових пошуком може розкладати запит на частини і у результаті знаходити багато результатів, які пов'язані лише з частиною запиту (запит 3). Пошук за підрядком поганий тим, що навіть незначна похибка у написанні повністю псує результати

## 4. Граф знань і GraphRAG

### 4.1 Побудова графа

In [76]:
import networkx as nx
import matplotlib.pyplot as plt
import community as community_louvain

G = nx.DiGraph()
for f in facts:
    for nm in (f["subject"], f["object"]):
        if nm not in G:
            G.add_node(nm, type=ENTITIES[nm]["type"], description=max(ENTITIES[nm]["descriptions"], key=len, default=""))
    rel = {"predicate": f["predicate"], "date": f["date"], "evidence": f["evidence"], "doc_id": f["doc_id"]}
    if G.has_edge(f["subject"], f["object"]):
        G[f["subject"]][f["object"]]["relations"].append(rel)
        G[f["subject"]][f["object"]]["weight"] += 1
    else:
        G.add_edge(f["subject"], f["object"], relations=[rel], weight=1)

U = nx.Graph()
U.add_nodes_from(G.nodes(data=True))
for u, v, d in G.edges(data=True):
    if U.has_edge(u, v):
        U[u][v]["weight"] += d["weight"]
    else:
        U.add_edge(u, v, weight=d["weight"])

print(f"Вузлів: {G.number_of_nodes()}, ребер: {G.number_of_edges()}, компонент зв'язності: {nx.number_connected_components(U)}")
top = sorted(U.degree, key=lambda x: -x[1])[:20]
print("20 перших вузлів з рівенем їх зв'язності:")
for n, d in top:
    print(f"\t{n} ({d})")

Вузлів: 249, ребер: 283, компонент зв'язності: 18
20 перших вузлів з рівенем їх зв'язності:
	Українська Центральна Рада (24)
	Симон Петлюра (24)
	Володимир Винниченко (22)
	Директорія УНР (21)
	Михайло Грушевський (20)
	Павло Скоропадський (16)
	Західноукраїнська Народна Республіка (16)
	Українська Держава (15)
	Українська Народна Республіка (14)
	Українська Національна Рада ЗУНР (13)
	Перший зимовий похід (9)
	Генеральний секретаріат (8)
	Український Народний Комітет (8)
	Студентський курінь Січових Стрільців (7)
	Київ (6)
	Львів (6)
	Гайдамацький кіш Слобідської України (6)
	Українська Галицька армія (5)
	Микола Міхновський (4)
	Українська демократично-хліборобська партія (4)


### 4.2 Спільноти (Louvain)

In [36]:
partition = community_louvain.best_partition(U, weight="weight", random_state=42)
communities = defaultdict(list)
for node, cid in partition.items():
    communities[cid].append(node)
communities = dict(sorted(communities.items(), key=lambda kv: -len(kv[1])))
print("Спільнот:", len(communities), "| модулярність:", round(community_louvain.modularity(partition, U, weight="weight"), 3))
for cid, members in list(communities.items())[:12]:
    best = sorted(members, key=lambda n: -U.degree(n))[:5]
    print(f"  #{cid:<2} розмір {len(members):<3} ключові: {', '.join(best)}")

Спільнот: 24 | модулярність: 0.738
  #11 розмір 34  ключові: Західноукраїнська Народна Республіка, Українська Національна Рада ЗУНР, Український Народний Комітет, Львів, Степан Баран
  #2  розмір 30  ключові: Директорія УНР, Українська Народна Республіка, Польща, Борис Мартос, Федір Швець
  #8  розмір 30  ключові: Симон Петлюра, Володимир Винниченко, Генеральний секретаріат, Революційна українська партія, Українська соціал-демократична робітнича партія
  #1  розмір 26  ключові: Українська Центральна Рада, Гайдамацький кіш Слобідської України, Харків, Сергій Єфремов, Мала Рада
  #4  розмір 24  ключові: Українська Держава, Київ, Микола Міхновський, Українська демократично-хліборобська партія, Перший Всеукраїнський військовий з'їзд
  #0  розмір 22  ключові: Михайло Грушевський, Україна, російські більшовики, Дон, Українська національно-демократична партія
  #12 розмір 20  ключові: Перший зимовий похід, Українська Галицька армія, Добровольча армія Денікіна, Армія УНР, Армія Української Нар

Отримано 24 спільноти, що більше ніж 18 компонентів зв'язності, що означає, що деякі компоненти на розбиті на групи. Рівень модулярності `0.738` досить близький до 1, що означає, що структура спільнот досить непогано виражена

### Візуалізація

Виконається інтерактивно через pyvis. Для перегляду потрібно відкрити файл `outputs/graph.html`

In [77]:
try:
    from pyvis.network import Network
    net = Network(height="800px", width="100%", directed=True, cdn_resources="in_line")
    for n, data in G.nodes(data=True):
        net.add_node(n, label=n, title=f"{data['type']}: {data['description']}",
                     color=f"hsl({(partition[n] * 47) % 360}, 65%, 60%)", size=10 + 3 * U.degree(n))
    for u, v, d in G.edges(data=True):
        label = ", ".join(sorted({r['predicate'] for r in d["relations"]}))
        net.add_edge(u, v, label=label, title="\n".join(f"{r['predicate']} {r['date'] or ''}: {r['evidence']}" for r in d["relations"]))
    net.show_buttons(filter_=["physics"])
    html = net.generate_html(notebook=False)
    (OUT / "graph.html").write_text(html, encoding="utf-8")
except Exception as e:
    import traceback; traceback.print_exc()

Граф:

![Full graph](./outputs/graph1.png)
![Part of a graph](./outputs/graph2.png)

### 4.3 Резюме спільнот
Для кожної достатньо великої спільноти формуємо промпт із переліком її сутностей і зв'язків та просимо LLM зробити короткий звіт. LLM бачить лише дані з графи конкретної спільноти, без вихідних статей. Тому узагальнення в резюме походять зі структури графа

In [78]:
MIN_COMMUNITY_SIZE = 3
MAX_COMMUNITIES    = 10

SYSTEM_SUMMARY = ("Ти — історик-аналітик. За переліком сутностей і зв'язків із графа знань склади стислий звіт про цю групу "
                  "(4–6 речень українською): яка тема/період її об'єднує, хто ключові учасники і яка їхня роль, "
                  "які головні зв'язки (союзи, конфлікти, підпорядкування), які закономірності видно. "
                  "Використовуй ЛИШЕ надані дані, нічого не вигадуй.")

def community_prompt(cid, members):
    mset = set(members)
    top_nodes = sorted(members, key=lambda n: -U.degree(n))[:25]
    node_lines = [f"- {n} [{G.nodes[n]['type']}]" + (f": {G.nodes[n]['description']}" if G.nodes[n]["description"] else "") for n in top_nodes]
    edge_lines = []
    for u, v, d in G.edges(data=True):
        if u in mset and v in mset:
            for r in d["relations"]:
                edge_lines.append(f"- {u} —{r['predicate']}→ {v}" + (f" ({r['date']})" if r["date"] else ""))
    return f"Спільнота #{cid}\n\nСутності:\n" + "\n".join(node_lines) + "\n\nЗв'язки:\n" + "\n".join(edge_lines[:50])

selected = [(cid, m) for cid, m in communities.items() if len(m) >= MIN_COMMUNITY_SIZE][:MAX_COMMUNITIES]
community_summaries = {}
for cid, members in selected:
    community_summaries[cid] = llm_text(community_prompt(cid, members), SYSTEM_SUMMARY)
    print(f"── Спільнота #{cid} ({len(members)} вузлів) ──\n{community_summaries[cid]}\n")

── Спільнота #11 (34 вузлів) ──
Ця спільнота об'єднана темою утворення та державного будівництва Західноукраїнської Народної Республіки (ЗУНР) у період 1918–1919 років. Ключовими організаційними структурами тут виступають Українська Національна Рада та Український Народний Комітет, які керували політичним життям та закладали підґрунтя для створення нової держави. Головними учасниками процесу є Євген Петрушевич (президент і диктатор), Сидор Голубович (очільник уряду — Державного секретаріату) та діячі Кость Левицький, Лонгин Цегельський і Степан Баран. Учасники тісно пов'язані відносинами членства й підпорядкування: Українська Національна Рада заснувала державні інституції та уряд, а делегати й члени комісій забезпечували перехід влади у Львові та дипломатичне представництво республіки. Закономірність цієї структури полягає в системному переході від внутрішньополітичного комітету до легітимних органів державної влади з чіткою ієрархією управління.

── Спільнота #2 (30 вузлів) ──
Ця груп

### 4.4 Отримання відповіді на глобальне питання через GraphRAG та Vector-RAG

Обрана запитання: `Хто є ключовими фігурами Української революції 1917–1921 рр. і як вони пов'язані між собою?`

Для GraphRAG передаємо всі наші резюме на основі 10 спільнот і просимо llm відповісти на запитання

Для VectorRAG виконуємо семантичний пошук передаючи запитання. З отриманих 10 найкращих чанків формуємо промпт та просимо llm відповісти на наше запитання 

In [67]:
GLOBAL_Q = next(q for q in QUESTIONS if q["id"] == "Q4")["text"]

# --- GraphRAG: reduce-крок ---
reduce_prompt = (f"Запитання: {GLOBAL_Q}\n\nНиже — резюме груп (спільнот) із графа знань.\n\n" +
                 "\n\n".join(f"[Група {i}] {community_summaries[cid]}" for i, (cid, _) in enumerate(selected, 1)) +
                 "\n\nДай повну, структуровану відповідь українською на основі цих резюме. Узагальни, виділи головні закономірності. "
                 "Не вигадуй фактів, яких немає в резюме.")
graphrag_answer = llm_text(reduce_prompt, "Ти — історик-аналітик. Відповідай точно та по суті.")

# --- Vector-RAG: top-N чанків -> LLM ---
hits = semantic_search(GLOBAL_Q, 10)
ctx = "\n\n".join(f"[Фрагмент {i}, {c['doc_title']}] {c['text']}" for i, (c, _) in enumerate(hits, 1))

vector_prompt = (f"Запитання: {GLOBAL_Q}\n\nФрагменти текстів:\n{ctx}\n\n"
                 "Дай повну відповідь українською ЛИШЕ на основі цих фрагментів. Якщо інформації бракує, скажи про це.")
vectorrag_answer = llm_text(vector_prompt, "Ти — історик-аналітик. Відповідай точно та по суті.")

print("ЗАПИТАННЯ:", GLOBAL_Q)
print("\n================ GraphRAG ================\n", graphrag_answer)
print("\n================ Vector-RAG ================\n", vectorrag_answer)

ЗАПИТАННЯ: Хто є ключовими фігурами Української революції 1917–1921 рр. і як вони пов'язані між собою?

================ GraphRAG ================
 На основі наданих даних із графа знань можна виділити ключові фігури Української революції 1917–1921 рр., які об’єднані в кілька державотворчих, військових та політичних осередків. 

### Ключові фігури та їх зв'язки за групами:

1. **Діячі Української Центральної Ради та УНР (1917–1918 рр.):**
   * **Ключові особи:** Володимир Винниченко, Симон Петлюра, Михайло Грушевський, Сергій Єфремов, Людмила Старицька-Черняхівська, Дмитро Антонович, Всеволод Голубович, Іван Стешенко, Христофор Барановський, Микола Стасюк, Павло Христюк.
   * **Зв'язки:** Вони очолювали вищі органи влади (Центральну Раду, Генеральний секретаріат, Малу Раду), партійні організації (РУП, УСДРП, УПСР) та військові комітети. Перебували у відносинах партійної та державної співпраці, проте між Винниченком і Петлюрою, а також іншими суб'єктами фіксувалося внутрішнє протистоянн

### Просте порівняння відповідей

In [79]:
def mentioned_entities(answer):
    low, hit = answer.lower(), set()
    for name, info in ENTITIES.items():
        if info["type"] == "Person":
            key = name.split()[-1].lower(); key = key[:max(4, len(key) - 2)]
        elif info["type"] == "Organization":
            key = name.lower()
        else:
            continue
        if len(key) > 3 and key in low:
            hit.add(name)
    return hit

used_nodes = set().union(*[set(m) for _, m in selected])
graph_docs = {f["doc_id"] for f in facts if f["subject"] in used_nodes and f["object"] in used_nodes}
vector_docs = {c["doc_id"] for c, _ in hits}

rows = [("Слів у відповіді", len(graphrag_answer.split()), len(vectorrag_answer.split())),
        ("Згадано осіб/організацій з графа", len(mentioned_entities(graphrag_answer)), len(mentioned_entities(vectorrag_answer))),
        ("Документів у вхідному контексті", len(graph_docs), len(vector_docs))]
print(f"{'показник':40} {'GraphRAG':>9} {'Vector-RAG':>11}")
for name, a, b in rows:
    print(f"{name:40} {a:>9} {b:>11}")

показник                                  GraphRAG  Vector-RAG
Слів у відповіді                               547         392
Згадано осіб/організацій з графа                76          21
Документів у вхідному контексті                 12           4


### Збереження відповіді за двома методами у файл `outputs/answers.json`

Окремо було збережено відповідь, яку повернули llm до файлу `outputs/answers.md` для кращого перегляду

In [81]:
(OUT / "answers.json").write_text(json.dumps({
    "question": GLOBAL_Q, "graphrag": graphrag_answer, "vector_rag": vectorrag_answer,
    "vector_chunks": [c["id"] for c, _ in hits], "community_summaries": {str(k): v for k, v in community_summaries.items()}
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("Збережено outputs/answers.json")

Збережено outputs/answers.json


### Висновки по GraphRAG та Vector-RAG

GraphRAG краще охоплює загальну структуру та взаємозв'язки між діячами, тоді як Vector-RAG краще відтворює конкретні біографічні факти та послідовність подій

GraphRAG має значно більше охоплення та показує зв'язки між цілими групами та навіть організаціями. В той час, як Vector-RAG акцентує основну увагу лише на 5 основних діячах, хоча й значно детальніше описує їх

Стосовно зв'язків, то GraphRAG загалом демонструє зв'язки на рівні приналежності до організації (загалом повторяє типи зв'язків, що визначені в сутностях), а Vector-RAG показує більш конкретні персональні зв'язки, оскільки інформація не обмежена чітко визначеним невеликим набором зв'язків сутностей

Через те, що GrapRAG побудований на основі сутностей, які ми отримали через LLM-екстракцію (як і факти), то в них можуть бути неточності, і вони є, що було продемонстровано раніше, тому деякі частини відповіді можуть бути некоректними, оскільки саме в структурі графу помилка. Vector-RAG такого не має, оскільки працює напряму з чанками (частинами оригінальних даних), проте оскільки це все працює через LLM, то помилки всеодно можуть виникати, проте можливо не так часто. Тобто на результати GrapRAG впливає якість побудови сутностей, а на Vector-RAG - якість розділення інформації на чанки (наприклад розмір). Але основні характеристики полягають в тому, що GraphRAG надто глобальний і модель може напривильно інтерпретувати велику модель, яка і так може мати помилки, а Vector-RAG - може не побачити ширшого контексту

## 5. Помилки та виправлення

Початково було використано модель `gemini-embedding-001` для ембедингу, проте як виявилося, ліміти даної моделі дуже швидко закінчилися. Як вирішення було вирішено перейти на локальну модель за допомогою ollama. Було обрано саме модель `bge-m3`, а не `nomic-embed-text` як вказано в умові, бо дана модель має кращу підтримку різних мов, що дуже важливо у даній роботі, де робота ведеться з українською, а не англійською мовою

Також проблеми були з візуалізацією графу, оскільки початково було вирішено намалювати його через matplotlib, але результат був досить складним для читання. Тому рішенням було використання інтерактивного підходу через pyvis

Початково використовувався лише перший опис на кожну сутність під час векторного пошуку, що було трохи менш ефективно, тому було вирішено використовувати найбільший зі списку, оскільки в ньому йомвірно більше інфорамції